In [98]:
import numpy as np
import pandas as pd

df = pd.read_csv('adult.csv')

In [99]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       32561 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education.num   32561 non-null  int64
 5   marital.status  32561 non-null  str  
 6   occupation      32561 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital.gain    32561 non-null  int64
 11  capital.loss    32561 non-null  int64
 12  hours.per.week  32561 non-null  int64
 13  native.country  32561 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [100]:
df.head(10)

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K
5,34,Private,216864,HS-grad,9,Divorced,Other-service,Unmarried,White,Female,0,3770,45,United-States,<=50K
6,38,Private,150601,10th,6,Separated,Adm-clerical,Unmarried,White,Male,0,3770,40,United-States,<=50K
7,74,State-gov,88638,Doctorate,16,Never-married,Prof-specialty,Other-relative,White,Female,0,3683,20,United-States,>50K
8,68,Federal-gov,422013,HS-grad,9,Divorced,Prof-specialty,Not-in-family,White,Female,0,3683,40,United-States,<=50K
9,41,Private,70037,Some-college,10,Never-married,Craft-repair,Unmarried,White,Male,0,3004,60,?,>50K


In [101]:
df.nunique()

age                  73
workclass             9
fnlwgt            21648
education            16
education.num        16
marital.status        7
occupation           15
relationship          6
race                  5
sex                   2
capital.gain        119
capital.loss         92
hours.per.week       94
native.country       42
income                2
dtype: int64

In [102]:
df.describe()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [103]:
def preprocessing(data):
    data["income"] = (data["income"] == ">50K").astype(int)
    # data['log_fnlwgt'] = np.log1p(data['fnlwgt'])
    data = data.drop("fnlwgt", axis=1)

    # data["log_capital.gain"] = np.log1p(data["capital.gain"])
    # data["log_capital.loss"] = np.log1p(data["capital.loss"])
    # data = data.drop(["capital.gain", "capital.loss"], axis=1)

    return data

In [104]:
from sklearn.model_selection import train_test_split


X = preprocessing(df)
y = X.pop('income')

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=0,
)

cat_features = X_train.select_dtypes(include=['object', 'str']).columns.tolist()

In [105]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score


model = CatBoostClassifier(
    iterations=1000,
    cat_features=cat_features,
    random_state=0,
    early_stopping_rounds=50,
    eval_metric="AUC",
    verbose=100,
    )

model.fit(X_train, y_train, eval_set=(X_valid, y_valid))

preds = model.predict_proba(X_valid)[:, 1]
score = roc_auc_score(y_valid, preds)

print(f"Accuracy: {score:.4f}")

Learning rate set to 0.070893
0:	test: 0.8821331	best: 0.8821331 (0)	total: 51.5ms	remaining: 51.5s
100:	test: 0.9211765	best: 0.9211765 (100)	total: 9.6s	remaining: 1m 25s
200:	test: 0.9247019	best: 0.9247057 (193)	total: 19.2s	remaining: 1m 16s
300:	test: 0.9256900	best: 0.9257061 (295)	total: 28.2s	remaining: 1m 5s
400:	test: 0.9262371	best: 0.9262371 (400)	total: 37.9s	remaining: 56.6s
500:	test: 0.9263130	best: 0.9263542 (497)	total: 47.8s	remaining: 47.6s
600:	test: 0.9266964	best: 0.9267061 (598)	total: 58.2s	remaining: 38.6s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.926898319
bestIteration = 630

Shrink model to first 631 iterations.
Accuracy: 0.9269


In [106]:
importance_features = model.feature_importances_
feature_names = model.feature_names_

features = pd.DataFrame({
    "Name": feature_names,
    "Importance": importance_features
})

features.sort_values(by="Importance", ascending=False)

,Name,Importance
9,capital.gain,20.922229
6,relationship,16.781718
0,age,13.388266
11,hours.per.week,8.481663
4,marital.status,8.215074
10,capital.loss,8.028701
5,occupation,7.284743
2,education,4.872624
3,education.num,4.583346
1,workclass,2.824148
